# Notebook 1 — Data Cleaning
### CDC Foodborne Disease Outbreak Surveillance System (1998–2015)

**Goal:** Transform the raw CDC dataset into a structurally clean, analysis-ready file.  
**Input:** `outbreaks.csv` — 19,119 records × 12 columns  
**Output:** `cleaned_data.csv` — 18,828 records × 9 columns

**Steps:**
1. Initial audit — shape, dtypes, missingness, duplicates
2. Drop irrelevant columns
3. Remove duplicates
4. Handle missing values (categorical → `'Unknown'`, numerical → `0.0`)
5. Final validation and export

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np

df_raw = pd.read_csv("data/outbreaks.csv")
print(f"Shape: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
df_raw.head(3)

## 2. Initial audit

Understanding what we have before making any changes.

In [ ]:
# Data types and non-null counts
df_raw.info()

In [ ]:
# Missingness — absolute counts and percentage
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(1)
audit = pd.DataFrame({"missing_n": missing, "missing_%": missing_pct,
                       "unique_values": df_raw.nunique()})
audit[audit["missing_n"] > 0].sort_values("missing_%", ascending=False)

In [ ]:
# Duplicate check
n_dupes = df_raw.duplicated().sum()
print(f"Duplicate rows: {n_dupes} ({n_dupes/len(df_raw)*100:.1f}% of dataset)")

**Audit summary:**

| Column | Missing | Decision |
|---|---|---|
| `Ingredient` | ~90% | Drop — too sparse to use |
| `Serotype/Genotype` | ~80% | Drop — outside EDA scope |
| `Status` | ~35% | Drop — administrative field |
| `Food` | 47% | Fill → `'Unknown'` |
| `Species` | 34% | Fill → `'Unknown'` |
| `Location` | 11% | Fill → `'Unknown'` |
| `Hospitalizations` | 19% | Fill → `0.0` (conservative) |
| `Fatalities` | 19% | Fill → `0.0` (conservative) |

**Missingness strategy rationale:** Replacing with `'Unknown'` rather than dropping rows
preserves ~47% of Food records and ~34% of Species records for temporal and geographic
analysis — where knowing *when* and *where* an outbreak occurred matters even if the
pathogen/food is unconfirmed.

## 3. Drop irrelevant columns

In [ ]:
df = df_raw.drop_duplicates().copy()
# Deduplicate on all original source fields BEFORE dropping metadata columns.
# Otherwise 11 distinct records become identical on the reduced 9-column schema.
df = df.drop(columns=["Ingredient", "Serotype/Genotype", "Status"])
print(f"Columns after drop: {df.columns.tolist()}")
print(f"Shape: {df.shape}")

## 4. Remove duplicates

In [ ]:
before = len(df_raw)
after = len(df)
# Deduplication was already applied to the full 12-column source above.
print(f"Removed: {before - after} duplicate rows  ({before:,} → {after:,})")

## 5. Handle missing values

In [ ]:
# Categorical columns → 'Unknown'
for col in ["Location", "Food", "Species"]:
    n_filled = df[col].isna().sum()
    df[col] = df[col].fillna("Unknown")
    print(f"{col}: filled {n_filled:,} missing values → 'Unknown'")

In [ ]:
# Severity fields: missing means unreported, NOT zero.
for col in ["Hospitalizations", "Fatalities"]:
    print(f"{col}: {df[col].isna().sum():,} missing values retained as NaN")

## 6. Final validation

In [ ]:
# Confirm zero missing values remain
assert df[["Year", "Month", "State", "Location", "Food", "Species", "Illnesses"]].notna().all().all()
assert len(df) == 18839, f"Unexpected row count: {len(df)}"
assert df["Hospitalizations"].isna().sum() == 3560
assert df["Fatalities"].isna().sum() == 3538
print("✓ Severity missingness intentionally retained (NaN)")
print(f"✓ Final shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print()
df.info()

In [ ]:
# Sanity check: key column distributions
print("Year range:  ", df["Year"].min(), "–", df["Year"].max())
print("States:      ", df["State"].nunique(), "unique values")
print("Food:        ", df["Food"].nunique(), "unique values (incl. 'Unknown')")
print("Species:     ", df["Species"].nunique(), "unique values (incl. 'Unknown')")
df.describe()

## 7. Export

In [ ]:
df.to_csv("data/cleaned_data.csv", index=False)
print("Saved: cleaned_data.csv")
print(f"  → {df.shape[0]:,} records × {df.shape[1]} columns")
print()
print("Next step: 02_entity_normalisation.ipynb")
print("  Food column:   3,128 unique strings → fuzzy grouping + manual mapping")
print("  Species column: 201 unique strings → fuzzy grouping + manual mapping")